# Library

In [5]:
pip install requests beautifulsoup4 pandas

Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install selenium webdriver-manager

Note: you may need to restart the kernel to use updated packages.


# Authors

## Get Data Detail Authors

In [ ]:
# Debugging Scrape Detail Profile (Static)

import requests
from bs4 import BeautifulSoup

# DEBUGGING
HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

def scrape_profile(url):
    res = requests.get(url, headers=HEADERS, timeout=30)
    res.raise_for_status()

    soup = BeautifulSoup(res.text, "html.parser")

    print("STATUS:", res.status_code)

    # ======================
    # SUBJECT
    # ======================
    subjects = []
    subject_section = soup.select_one("div.profile-subject ul.subject-list")

    if subject_section:
        for a in subject_section.select("li a"):
            subjects.append(a.get_text(strip=True))
    else:
        print("❌ Subject section NOT FOUND")

    # ======================
    # METRICS TABLE
    # ======================
    metrics = {}

    rows = soup.select("tbody tr")
    print("Total rows found:", len(rows))

    for row in rows:
        cols = row.find_all("td")
        if len(cols) >= 3:
            label = cols[0].get_text(strip=True)
            scopus = cols[1].get_text(strip=True)
            gscholar = cols[2].get_text(strip=True)

            metrics[f"{label}_Scopus"] = scopus
            metrics[f"{label}_GScholar"] = gscholar

            print(f"{label} → Scopus:{scopus}, GScholar:{gscholar}")

    return {
        "subjects": subjects,
        "metrics": metrics
    }


if __name__ == "__main__":
    url = "https://sinta.kemdiktisaintek.go.id/authors/profile/257962"
    result = scrape_profile(url)

    print("\n=== FINAL RESULT ===")
    print(result)


In [ ]:
# Get Data Profile URL Unikom Sinta

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

BASE_URL = "https://sinta.kemdiktisaintek.go.id/affiliations/authors/528/"

data_local= pd.read_csv("Tabel Authors Local.csv")

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

def scrape_page(page):
    url = f"{BASE_URL}?page={page}"
    print(f"Scraping page {page}")

    r = requests.get(url, headers=HEADERS)
    if r.status_code != 200:
        return []

    soup = BeautifulSoup(r.text, "html.parser")
    profiles = soup.select("div.col-lg")

    results = []

    for p in profiles:
        name_div = p.find("div", class_="profile-name")
        if not name_div:
            continue

        a_tag = name_div.find("a")
        name = a_tag.get_text(strip=True) if a_tag else name_div.get_text(strip=True)
        profile_link = a_tag["href"] if a_tag and a_tag.get("href") else None

        dept_div = p.find("div", class_="profile-dept")
        dept = dept_div.get_text(strip=True) if dept_div else None

        id_div = p.find("div", class_="profile-id")
        sinta_id = id_div.get_text(strip=True).replace("ID :", "").strip() if id_div else None

        scopus_h = gs_h = None
        hindex_div = p.find("div", class_="profile-hindex")
        if hindex_div:
            for span in hindex_div.find_all("span"):
                txt = span.get_text(strip=True)
                if "Scopus" in txt:
                    scopus_h = txt.split(":")[-1].strip()
                elif "GS" in txt:
                    gs_h = txt.split(":")[-1].strip()

        sinta_3yr = sinta = affil_3yr = affil = None
        for col in p.select("div.col"):
            label = col.find("div", class_="stat-text")
            value = col.find("div", class_="stat-num")
            if not label or not value:
                continue

            l = label.get_text(strip=True)
            v = value.get_text(strip=True)

            if l == "SINTA Score 3Yr":
                sinta_3yr = v
            elif l == "SINTA Score":
                sinta = v
            elif l == "Affil Score 3Yr":
                affil_3yr = v
            elif l == "Affil Score":
                affil = v

        # nidn diambil dari data_local dengan mencocokkan nama
        nidn = None
        for _, row in data_local.iterrows():
            if row["nama"].strip().lower() == name.strip().lower():
                nidn = row["nidn"]
                break
            else:
                nidn = "Tidak Diketahui"

        # faculty diambil dari data_local dengan mencocokkan nama
        faculty = None
        for _, row in data_local.iterrows():
            if row["nama"].strip().lower() == name.strip().lower():
                faculty = row["Fakultas"]
                break
            else:
                faculty = "Tidak Diketahui"

        results.append({
            "fullname": name,
            "id_sinta": sinta_id,
            "nidn": nidn,
            "profile_url": profile_link,
            "departemen": dept,
            "faculty": faculty,
            "s_hindex_scopus": scopus_h,
            "s_hindex_gscholar": gs_h,
            "sinta_score_3yr": sinta_3yr,
            "sinta_score_overall": sinta,
            "affil_score": affil,
            "affil_score_3yr": affil_3yr,
        })

    return results


# # =====================
# # PAGING
# # =====================
all_data = []
page = 1

while True:
    data = scrape_page(page)
    if len(data) == 0:
        print("Data habis")
        break

    all_data.extend(data)
    page += 1
    time.sleep(1)

df = pd.DataFrame(all_data)
df.to_csv("Tabel Authors Unikom Sinta.csv", index=False, encoding="utf-8-sig")

print("Total data:", len(df))
print(df.head())


In [ ]:
# Running Full Scraping Lanjutan Detail Profile

import pandas as pd
import requests
from bs4 import BeautifulSoup
import time

dfAuthorUnikom = pd.read_csv("Tabel Authors Unikom Sinta.csv")

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

COLUMN_MAPPING_SCOPUS = {
    "Article": "s_article_scopus",
    "Citation": "s_citation_scopus",
    "Cited Document": "s_cited_document_scopus",
    "i10-Index": "s_i10_index_scopus",
    "G-Index": "s_gindex_scopus",
    "h-index": "s_hindex_scopus",
}

COLUMN_MAPPING_GSCHOLAR = {
    "Article": "s_article_gscholar",
    "Citation": "s_citation_gscholar",
    "Cited Document": "s_cited_document_gscholar",
    "i10-Index": "s_i10_index_gscholar",
    "G-Index": "s_gindex_gscholar",
    "h-index": "s_hindex_gscholar",
}

def scrape_profile(url):
    res = requests.get(url, headers=HEADERS, timeout=30)
    res.raise_for_status()

    soup = BeautifulSoup(res.text, "html.parser")

    # ======================
    # SUBJECT
    # ======================
    subjects = []
    subject_section = soup.select_one("div.profile-subject ul.subject-list")
    if subject_section:
        for a in subject_section.select("li a"):
            subjects.append(a.get_text(strip=True))

    # ======================
    # METRICS TABLE
    # ======================
    metrics = {}

    rows = soup.select("tbody tr")
    for row in rows:
        cols = row.find_all("td")
        if len(cols) >= 3:
            label = cols[0].get_text(strip=True)
            scopus = cols[1].get_text(strip=True)
            gscholar = cols[2].get_text(strip=True)  

            # gunakan mapping custom
            label_clean_scopus = COLUMN_MAPPING_SCOPUS.get(
                label,
                label.lower().replace(" ", "_")
            )

            label_clean_gscholar = COLUMN_MAPPING_GSCHOLAR.get(
                label,
                label.lower().replace(" ", "_")
            )

            metrics[f"{label_clean_scopus}"] = scopus
            metrics[f"{label_clean_gscholar}"] = gscholar

    return {
        "subject_research": "; ".join(subjects),
        **metrics
    }


def main():
    results = []

    for _, row in dfAuthorUnikom.iterrows():
        url = row["profile_url"]
        print(f"Scraping: {url}")

        try:
            data = scrape_profile(url)

            merged = {**row.to_dict(), **data}
            merged = {k: "" if v is None else str(v) for k, v in merged.items()}

            results.append(merged)

        except Exception as e:
            print(f"Failed: {url} | {e}")
            error_row = row.to_dict()
            error_row["error"] = str(e)
            results.append({k: str(v) for k, v in error_row.items()})

        time.sleep(2)

    output_df = pd.DataFrame(results, dtype=str)
    output_df.to_csv(
        "Tabel Authors Unikom Sinta.csv",
        index=False,
        encoding="utf-8"
    )


if __name__ == "__main__":
    main()


## Get Data Authors Index

In [ ]:
# Debugging Chart Data

from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.edge.options import Options
from webdriver_manager.microsoft import EdgeChromiumDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

options = Options()
options.add_argument("--headless=new")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")

# try to install driver via webdriver_manager; if offline, fallback to local msedgedriver (must be in PATH)
try:
    service = Service(EdgeChromiumDriverManager().install())
    driver = webdriver.Edge(service=service, options=options)
except Exception as e:
    print("webdriver_manager failed, trying local msedgedriver (must be in PATH):", e)
    driver = webdriver.Edge(options=options)

driver.get("https://sinta.kemdiktisaintek.go.id/authors/profile/258671")

# wait for the chart element to be present (avoid fixed sleep)
try:
    WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.ID, "quartile-pie")))
except Exception as e:
    print("quartile-pie element not found or timed out:", e)

quartile_data = driver.execute_script("""
var el = document.getElementById("quartile-pie");
if (!el) return null;
var chart = (typeof echarts !== 'undefined') ? echarts.getInstanceByDom(el) : null;
if (!chart) return null;
return chart.getOption().series[0].data;
""")

print(quartile_data)
driver.quit()


In [ ]:
# Full Scraping Lanjutan Detail Profile + Chart Data

import pandas as pd
import requests
from bs4 import BeautifulSoup
import time

from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.edge.options import Options
from webdriver_manager.microsoft import EdgeChromiumDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# ======================
# CONFIG
# ======================
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

WAIT_TIME = 15


# ======================
# SELENIUM INIT (EDGE)
# ======================
def init_driver():
    options = Options()
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")

    try:
        service = Service(EdgeChromiumDriverManager().install())
        driver = webdriver.Edge(service=service, options=options)
    except Exception:
        driver = webdriver.Edge(options=options)

    return driver



# ======================
# SCRAPE QUARTILE PIE (ECHARTS)
# ======================
def scrape_quartile_pie(driver):
    result = {
        "mount_Q1": 0,
        "mount_Q2": 0,
        "mount_Q3": 0,
        "mount_Q4": 0,
        "mount_No-Q": 0
    }

    try:
        # tunggu sampai ECHARTS INSTANCE ADA
        WebDriverWait(driver, 20).until(
            lambda d: d.execute_script("""
                var el = document.getElementById("quartile-pie");
                if (!el || typeof echarts === 'undefined') return false;
                return echarts.getInstanceByDom(el) !== null;
            """)
        )

        quartile_data = driver.execute_script("""
            var el = document.getElementById("quartile-pie");
            var chart = echarts.getInstanceByDom(el);
            return chart.getOption().series[0].data;
        """)

        for q in quartile_data:
            name = q.get("name", "").strip()
            value = q.get("value", 0)

            if name in ["Q1", "Q2", "Q3", "Q4"]:
                result[f"mount_{name}"] = value
            else:
                result["mount_No-Q"] += value

    except Exception as e:
        print("Quartile gagal:", e)

    return result


# ======================
# MAIN
# ======================
def main():
    results = []
    driver = init_driver()

    for _, row in dfAuthorUnikom.iterrows():
        url = row["profile_url"]
        print(f"Scraping: {url}")

        try:
            # profile_data = scrape_profile(url)
            driver.get(url)
            quartile_data = scrape_quartile_pie(driver)

            merged = {
                **row.to_dict(),
                **quartile_data
            }

            # normalisasi nilai kosong
            merged = {
                k: "" if v in [None, "nan", "NaN"] else str(v)
                for k, v in merged.items()
            }

            results.append(merged)

        except Exception as e:
            print(f"❌ Gagal total: {url} | {e}")
            error_row = row.to_dict()
            error_row["error"] = str(e)
            results.append({k: str(v) for k, v in error_row.items()})

        time.sleep(2)

    driver.quit()

    output_df = pd.DataFrame(results)
    output_df.to_csv(
        "Tabel Authors Unikom Sinta.csv",
        index=False,
        encoding="utf-8"
    )


if __name__ == "__main__":
    main()


# Cleaning

In [ ]:
FACULTY_MAPPING = {
    "Fakultas Teknik dan Ilmu Komputer": [
        "Teknik Informatika",
        "Sistem Informasi",
        "Sistem Komputer",
        "Teknik Komputer",
        "Teknik Elektro",
        "Teknik Industri",
        "Teknik Sipil",
        "Teknik Arsitektur",
        "Teknik Perencanaan Wilayah Dan Kota",
        "Manajemen Informatika",
        "Komputerisasi Akuntansi",
    ],

    "Fakultas Ekonomi dan Bisnis": [
        "Manajemen",
        "Akuntansi",
        "Manajemen Pemasaran",
        "Keuangan Dan Perbankan",
        
    ],

    "Fakultas Ilmu Sosial dan Ilmu Politik": [
        "Ilmu Komunikasi",
        "Ilmu Pemerintahan",
        "Ilmu Hubungan Internasional",
    ],

    "Fakultas Sastra": [
        "Sastra Inggris",
        "Sastra Jepang",
    ],

    "Fakultas Desain": [
        "Desain Interior",
        "Desain Komunikasi Visual",
        "Desain Grafis",
        "Desain"
    ],

    "Fakultas Hukum": [
        "Ilmu Hukum",
    ],
}
